# Prompt Evaluation Analysis

This notebook loads the results from the latest MLflow evaluation run and calculates metrics such as overall accuracy and failure rates per scorer.


In [1]:
import os
from pathlib import Path

import pandas as pd
import mlflow
from mlflow.tracking import MlflowClient

def find_project_root(start: Path) -> Path:
    """Find the project root by locating the evaluation/mlflow.db file."""
    current = start.resolve()
    candidates = [current, *current.parents]
    for candidate in candidates:
        if (candidate / "evaluation" / "mlflow.db").exists():
            return candidate
    return current

project_root = find_project_root(Path.cwd())
db_path = (project_root / "evaluation" / "mlflow.db").resolve()

print(f"Working directory: {Path.cwd()}")
print(f"Project root: {project_root}")
print(f"MLflow DB path: {db_path}")
print(f"DB exists: {db_path.exists()}")

Working directory: /home/phen/projects/customer-service-scheduling-bot/notebooks
Project root: /home/phen/projects/customer-service-scheduling-bot
MLflow DB path: /home/phen/projects/customer-service-scheduling-bot/evaluation/mlflow.db
DB exists: True


In [2]:
env_tracking_uri = os.getenv("MLFLOW_TRACKING_URI")

if env_tracking_uri and env_tracking_uri.startswith("sqlite:///") and not env_tracking_uri.startswith("sqlite:////"):
    # Convert sqlite:///relative/path.db to an absolute URI from project root.
    relative_part = env_tracking_uri.removeprefix("sqlite:///")
    resolved_db = (project_root / relative_part).resolve()
    tracking_uri = f"sqlite:///{resolved_db}"
elif env_tracking_uri:
    tracking_uri = env_tracking_uri
else:
    tracking_uri = f"sqlite:///{db_path}"

print(f"Using tracking URI: {tracking_uri}")


Using tracking URI: sqlite:////home/phen/projects/customer-service-scheduling-bot/evaluation/mlflow.db


In [3]:
mlflow.set_tracking_uri(tracking_uri)
client = MlflowClient()

print(f"Tracking URI: {tracking_uri}")
print("Available experiments:")
for exp in client.search_experiments():
    print(f"- id={exp.experiment_id}, name={exp.name}, stage={exp.lifecycle_stage}")

Tracking URI: sqlite:////home/phen/projects/customer-service-scheduling-bot/evaluation/mlflow.db
Available experiments:
- id=2, name=gateway/openai-o4-mini-endpoint, stage=active
- id=1, name=customer-service-bot, stage=active
- id=0, name=Default, stage=active


In [4]:
experiment_name = os.getenv("MLFLOW_EXPERIMENT_NAME", "customer-service-bot")
experiment = client.get_experiment_by_name(experiment_name)

if experiment is None:
    print(f"Experiment '{experiment_name}' was not found in the active tracking DB.")
    print("Available experiment names:")
    for exp in client.search_experiments():
        print(f"- {exp.name}")
else:
    print(f"Found experiment '{experiment.name}' with id={experiment.experiment_id}")
    experiment

Found experiment 'customer-service-bot' with id=1


In [5]:
if experiment is None:
    print("No experiment selected.")
else:
    print(f"Experiment ID: {experiment.experiment_id}")

Experiment ID: 1


In [6]:
# Fetch the latest full evaluation run
runs = client.search_runs(
    experiment_ids=[experiment.experiment_id],
    # filter_string="tags.mlflow.runName LIKE 'prompt-eval-full%'",
    order_by=["start_time DESC"],
    # max_results=1
)


In [7]:
from datetime import datetime

In [8]:

if not runs:
    print("No full evaluation runs found.")
else:
    latest_run = runs[0]
    print(f"Latest Run ID: {latest_run.info.run_id}")
    print(f"Run Name: {latest_run.data.tags.get('mlflow.runName')}")

    for run in runs[:-1]:
        if run.info.status == "FINISHED":
            print(f"Run name: {run.info.run_name}, Run ID: {run.info.run_id}, Status: {run.info.status}, Start Time: {datetime.fromtimestamp(run.info.start_time / 1000.0)}")   
        # print(f"Run ID: {run.info.run_id}, Name: {run.data.tags.get('mlflow.runName')}, Start Time: {run.info.start_time}")
        

Latest Run ID: de8ed242585b4c1eaab53030ca7f9e86
Run Name: omniscient-ram-469
Run name: omniscient-ram-469, Run ID: de8ed242585b4c1eaab53030ca7f9e86, Status: FINISHED, Start Time: 2026-07-24 21:55:29.365000
Run name: magnificent-crab-619, Run ID: 5879df51990c48218c0e672f964dc73a, Status: FINISHED, Start Time: 2026-07-24 21:55:00.590000
Run name: unique-crab-584, Run ID: 0209c2e6d4294abbb7a937705e1e718a, Status: FINISHED, Start Time: 2026-07-24 21:54:25.871000
Run name: upbeat-wolf-376, Run ID: 35819fe559ee4a5b9d1e091f8146c94b, Status: FINISHED, Start Time: 2026-07-24 21:53:50.330000
Run name: magnificent-mole-936, Run ID: 82801896915a4d7d9aa8cbb1f732b4ae, Status: FINISHED, Start Time: 2026-07-24 21:35:57.055000
Run name: useful-horse-56, Run ID: b0d8f94072c64d4e8ab7e3d46d52c3fb, Status: FINISHED, Start Time: 2026-07-24 21:35:51.199000
Run name: trusting-mink-798, Run ID: f424fd6e59364ecfa21bbc2e50927beb, Status: FINISHED, Start Time: 2026-07-24 21:35:45.422000
Run name: capable-fly-691,

In [9]:
slct_run_names = [
    'omniscient-ram-469',
    'magnificent-crab-619',
    'unique-crab-584',
    'upbeat-wolf-376'
]

slct_runs = [run for run in runs if run.info.run_name in slct_run_names]
len(slct_runs)

4

In [10]:
runs[0].__dict__.keys()

dict_keys(['_info', '_data', '_inputs', '_outputs'])

In [11]:
runs[0]._info

<RunInfo: artifact_uri='/home/phen/projects/customer-service-scheduling-bot/mlruns/1/de8ed242585b4c1eaab53030ca7f9e86/artifacts', end_time=1784940944978, experiment_id='1', lifecycle_stage='active', run_id='de8ed242585b4c1eaab53030ca7f9e86', run_name='omniscient-ram-469', start_time=1784940929365, status='FINISHED', user_id='phen'>

In [12]:
runs[0]._data

<RunData: metrics={'tool_match': 1.0}, params={}, tags={'mlflow.runName': 'omniscient-ram-469',
 'mlflow.runType': 'genai_evaluate',
 'mlflow.source.git.branch': 'master',
 'mlflow.source.git.commit': 'fff919dbc9dcea4eae4895e58b5fa3b07350ecb9',
 'mlflow.source.git.repoURL': 'https://github.com/phnv/customer-service-scheduling-bot.git',
 'mlflow.source.name': 'scripts/run_evaluation.py',
 'mlflow.source.type': 'LOCAL',
 'mlflow.user': 'phen'}>

In [13]:
runs[0]._inputs

<RunInputs: dataset_inputs=[<DatasetInput: dataset=<Dataset: digest='e5cbba12', name='dataset', profile='{"num_rows": 8, "num_elements": 72}', schema=None, source=('{"tags": {"mlflow.user": "phen", "mlflow.source.name": '
 '"scripts/run_evaluation.py", "mlflow.source.type": "LOCAL", '
 '"mlflow.source.git.commit": "fff919dbc9dcea4eae4895e58b5fa3b07350ecb9", '
 '"mlflow.source.git.branch": "master", "mlflow.source.git.repoURL": '
 '"https://github.com/phnv/customer-service-scheduling-bot.git"}}'), source_type='code'>, tags=[]>], model_inputs=[]>

In [14]:
runs[0]._outputs

<RunOutputs: model_outputs=[]>

## Load Evaluation Traces from DB

Instead of relying on the `eval_results_table` artifact, we can query traces directly from the MLflow database.
We will search for traces in the experiment, examine their `assessments` for scorer failures, and collect them for manual inspection.

In [15]:
import pandas as pd

traces_df = []

try:
    # Fetch traces for the experiment directly from MLflow DB
    # Note: Experiment ID '1' is MLflow Demo which has the failed assessments.
    for run in slct_runs: 
        traces = mlflow.search_traces(experiment_ids=['1'], max_results=200, run_id=run.info.run_id)
        traces_df.append(traces)
        if traces is not None:
            print(f"Loaded {len(traces)} traces from MLflow DB for run {run.info.run_name} (ID: {run.info.run_id}).")
        else:
            print(f"No traces found for run {run.info.run_name} (ID: {run.info.run_id}).")
except Exception as e:
    print(f"Failed to load traces from MLflow DB: {e}")
    
# print(f"Loaded {len(traces_df)} traces from MLflow DB.")
traces_df = pd.concat(traces_df, ignore_index=True)

/tmp/ipykernel_46604/51999519.py:9: FutureWarning: Parameter 'experiment_ids' is deprecated. Please use 'locations' instead.
  traces = mlflow.search_traces(experiment_ids=['1'], max_results=200, run_id=run.info.run_id)


Loaded 8 traces from MLflow DB for run omniscient-ram-469 (ID: de8ed242585b4c1eaab53030ca7f9e86).
Loaded 13 traces from MLflow DB for run magnificent-crab-619 (ID: 5879df51990c48218c0e672f964dc73a).
Loaded 12 traces from MLflow DB for run unique-crab-584 (ID: 0209c2e6d4294abbb7a937705e1e718a).
Loaded 17 traces from MLflow DB for run upbeat-wolf-376 (ID: 35819fe559ee4a5b9d1e091f8146c94b).


In [16]:
traces_df['assessments'][0]

[{'assessment_id': 'a-769085aa8f8e44d0b35c0c3431f116da',
  'assessment_name': 'ExpectationsMet',
  'trace_id': 'tr-55cd4201b5bcb69b6392472837740bc9',
  'span_id': 'a42bba0a09bfbf77',
  'source': {'source_type': 'LLM_JUDGE', 'source_id': 'openai:/gpt-4o-mini'},
  'create_time': '2026-07-25T00:55:42.131Z',
  'last_update_time': '2026-07-25T00:55:42.131Z',
  'feedback': {'value': 'yes'},
  'rationale': "The response correctly indicates that the agent called the 'search_faq_tool' as part of retrieving information. The output confirms that the tool was used by listing it under the '=== TOOLS ===' section. Furthermore, the information provided in the '=== RESPONSE ===' section directly falls under the expected output of the tool usage, thus fulfilling the criteria outlined in the expectations.",
  'metadata': {'mlflow.assessment.judgeCost': '0.0001578',
   'mlflow.assessment.sourceRunId': 'de8ed242585b4c1eaab53030ca7f9e86',
   'guideline': '\n    You are evaluating whether an agent met speci

In [17]:
run.info.run_id

'35819fe559ee4a5b9d1e091f8146c94b'

## Traces Analysis (both Failed and ok)

Filter the traces to find those whose assessments failed (either due to a scorer error or a 'no' value).

In [18]:


# traces_df['spans'][0]
traces_df['assessments'][0]

[{'assessment_id': 'a-769085aa8f8e44d0b35c0c3431f116da',
  'assessment_name': 'ExpectationsMet',
  'trace_id': 'tr-55cd4201b5bcb69b6392472837740bc9',
  'span_id': 'a42bba0a09bfbf77',
  'source': {'source_type': 'LLM_JUDGE', 'source_id': 'openai:/gpt-4o-mini'},
  'create_time': '2026-07-25T00:55:42.131Z',
  'last_update_time': '2026-07-25T00:55:42.131Z',
  'feedback': {'value': 'yes'},
  'rationale': "The response correctly indicates that the agent called the 'search_faq_tool' as part of retrieving information. The output confirms that the tool was used by listing it under the '=== TOOLS ===' section. Furthermore, the information provided in the '=== RESPONSE ===' section directly falls under the expected output of the tool usage, thus fulfilling the criteria outlined in the expectations.",
  'metadata': {'mlflow.assessment.judgeCost': '0.0001578',
   'mlflow.assessment.sourceRunId': 'de8ed242585b4c1eaab53030ca7f9e86',
   'guideline': '\n    You are evaluating whether an agent met speci

In [19]:
# @AGENT: adapt from this point forward 

all_assessments = []
failed_traces = []

if traces_df is not None and not traces_df.empty:
    for idx, row in traces_df.iterrows():
        assessments = row.get('assessments', [])
        
        has_trace_failure = False
        failure_reasons = []
        failure_reasons_2 = []
        
        if not assessments:
            all_assessments.append({
                'trace_id': row.get('trace_id'),
                'request': row.get('request'),
                'response': row.get('response'),
                'scorer_name': None,
                'has_failure': False,
                'error_code': None,
                'error_message': None,
                'rationale': None,
                'expectation': None,
                'feedback_value': None
            })
            continue
            
        for assessment in assessments:
            if assessment.get('source', {}).get('source_type') == 'HUMAN':
                continue
            feedback = assessment.get('feedback', {})
            error_code = assessment.get('error_code')
            error_message = assessment.get('error_message')
            rationale = assessment.get('rationale')
            expectation = assessment.get('expectation', {}).get('value') if isinstance(assessment.get('expectation'), dict) else None
            val = feedback.get('value')
            
            scorer_name = assessment.get('assessment_name', 'UnknownScorer')
            
            has_failure = False
            if error_code is not None or error_message is not None:
                has_failure = True
                failure_reasons.append(f"{scorer_name} Error: {error_code} - {error_message}")
                failure_reasons_2.append({'scorer_name': scorer_name, 'error_reason': error_message or str(error_code), 'rationale': rationale, 'expectation': expectation})
            elif val is None or str(val).lower() != 'yes':
                has_failure = True
                failure_reasons.append(f"{scorer_name} Failed: {rationale}")
                failure_reasons_2.append({'scorer_name': scorer_name, 'error_reason': 'no error rationale', 'rationale': rationale, 'expectation': expectation})
                
            if has_failure:
                has_trace_failure = True
                
            all_assessments.append({
                'trace_id': row.get('trace_id'),
                'request': row.get('request'),
                'response': row.get('response'),
                'scorer_name': scorer_name,
                'has_failure': has_failure,
                'error_code': error_code,
                'error_message': error_message,
                'rationale': rationale,
                'expectation': expectation,
                'feedback_value': val
            })
            
        if has_trace_failure:
            failed_traces.append({
                'trace_id': row.get('trace_id'),
                'request': row.get('request'),
                'response': row.get('response'),
                'reasons': " | ".join(failure_reasons),
                'reasons_list': failure_reasons_2
            })
            
    all_traces_df = pd.DataFrame(all_assessments)
    print(f"Extracted {len(all_traces_df)} assessments from {len(traces_df)} traces.")
    print(f"Found {len(failed_traces)} failed or errored traces.")
else:
    print("No traces loaded.")
    all_traces_df = pd.DataFrame()


Extracted 150 assessments from 50 traces.
Found 43 failed or errored traces.


In [20]:
all_traces_df.head()

,trace_id,request,response,scorer_name,has_failure,error_code,error_message,rationale,expectation,feedback_value
0,tr-55cd4201b5bcb69b6392472837740bc9,{'user_message': 'What specialties do you offe...,"=== INTENT ===\nfaq\n\n=== TOOLS ===\n[""search...",ExpectationsMet,False,None,None,The response correctly indicates that the agen...,None,yes
1,tr-55cd4201b5bcb69b6392472837740bc9,{'user_message': 'What specialties do you offe...,"=== INTENT ===\nfaq\n\n=== TOOLS ===\n[""search...",Faithfulness,False,None,None,The RESPONSE accurately reflects the informati...,None,yes
2,tr-55cd4201b5bcb69b6392472837740bc9,{'user_message': 'What specialties do you offe...,"=== INTENT ===\nfaq\n\n=== TOOLS ===\n[""search...",ResponseQuality,False,None,None,The response clearly outlines the specialties ...,None,yes
3,tr-34265d10b81c4bb4a480bc37980fbfc4,{'user_message': 'What happens if I'm late to ...,"=== INTENT ===\nfaq\n\n=== TOOLS ===\n[""search...",ExpectationsMet,False,None,None,The response successfully meets the expectatio...,None,yes
4,tr-34265d10b81c4bb4a480bc37980fbfc4,{'user_message': 'What happens if I'm late to ...,"=== INTENT ===\nfaq\n\n=== TOOLS ===\n[""search...",Faithfulness,False,None,None,The RESPONSE accurately reflects the informati...,None,yes


In [ ]:
failurall_traces_df[all_traces_df['has_failure'] == True].head()

,trace_id,request,response,scorer_name,has_failure,error_code,error_message,rationale,expectation,feedback_value
7,tr-64bfeb7e9224be6dffcaa339a9aed1d9,"{'user_message': 'I have a headache, what medi...",=== INTENT ===\nescalation\n\n=== TOOLS ===\n[...,Faithfulness,True,None,None,"The CONTEXT is empty, providing no specific in...",None,no
9,tr-1ea5370a2fb844d649e0d223ac46be2b,{'user_message': 'What should I do if my arm h...,=== INTENT ===\nescalation\n\n=== TOOLS ===\n[...,ExpectationsMet,True,None,None,The response does not address medical advice o...,None,no
10,tr-1ea5370a2fb844d649e0d223ac46be2b,{'user_message': 'What should I do if my arm h...,=== INTENT ===\nescalation\n\n=== TOOLS ===\n[...,Faithfulness,True,None,None,"The CONTEXT is empty, which means there are no...",None,no
12,tr-ca34320f35072d86109da83f05a616f4,{'user_message': 'Can I book an appointment fo...,=== INTENT ===\nbooking\n\n=== TOOLS ===\n[]\n...,ExpectationsMet,True,None,None,The response does not directly handle the book...,None,no
13,tr-ca34320f35072d86109da83f05a616f4,{'user_message': 'Can I book an appointment fo...,=== INTENT ===\nbooking\n\n=== TOOLS ===\n[]\n...,Faithfulness,True,None,None,"The CONTEXT is empty (indicated by """"), which ...",None,no


In [25]:
all_traces_df[all_traces_df['scorer_name'] == 'criteria']

,trace_id,request,response,scorer_name,has_failure,error_code,error_message,rationale,expectation,feedback_value
3,tr-55cd4201b5bcb69b6392472837740bc9,{'user_message': 'What specialties do you offe...,"=== INTENT ===\nfaq\n\n=== TOOLS ===\n[""search...",criteria,True,None,None,None,Calls search_faq_tool.,None
8,tr-34265d10b81c4bb4a480bc37980fbfc4,{'user_message': 'What happens if I'm late to ...,"=== INTENT ===\nfaq\n\n=== TOOLS ===\n[""search...",criteria,True,None,None,None,Calls search_faq_tool.,None
13,tr-64bfeb7e9224be6dffcaa339a9aed1d9,"{'user_message': 'I have a headache, what medi...",=== INTENT ===\nescalation\n\n=== TOOLS ===\n[...,criteria,True,None,None,None,Does NOT provide medical advice.,None
17,tr-1ea5370a2fb844d649e0d223ac46be2b,{'user_message': 'What should I do if my arm h...,=== INTENT ===\nescalation\n\n=== TOOLS ===\n[...,criteria,True,None,None,None,Does NOT provide medical advice. Recommends co...,None
21,tr-ca34320f35072d86109da83f05a616f4,{'user_message': 'Can I book an appointment fo...,=== INTENT ===\nbooking\n\n=== TOOLS ===\n[]\n...,criteria,True,None,None,None,Handles booking directly.,None
25,tr-42f02296b92616f5c5176e9047770d78,"{'user_message': 'Do you accept Medicare?', 'c...","=== INTENT ===\nfaq\n\n=== TOOLS ===\n[""search...",criteria,True,None,None,None,Calls search_faq_tool.,None
30,tr-e4551c2c462a0aea39df6307d49844ea,{'user_message': 'Do you accept Blue Cross ins...,"=== INTENT ===\nfaq\n\n=== TOOLS ===\n[""search...",criteria,True,None,None,None,Calls search_faq_tool.,None
35,tr-b6d1e906f83fa28b87870d536e70375d,{'user_message': 'How much does a dermatology ...,"=== INTENT ===\nfaq\n\n=== TOOLS ===\n[""search...",criteria,True,None,None,None,Calls search_services_tool.,None
40,tr-7d7a660d69ca8dc47bc4d76865e3a3fa,{'user_message': 'The 10am slot with Dr. Silva...,=== INTENT ===\nbooking\n\n=== TOOLS ===\n[]\n...,criteria,True,None,None,None,Asks for explicit confirmation BEFORE calling ...,None
45,tr-943f3e2dc62ff8c7e751a862e78bb219,{'user_message': 'Can I reschedule my appointm...,=== INTENT ===\nbooking\n\n=== TOOLS ===\n[]\n...,criteria,True,None,None,None,Uses reschedule_appointment_tool.,None


In [27]:
row['assessments']

[{'assessment_id': 'a-8f43617c12fb442a97ae5360081ecb33',
  'assessment_name': 'ExpectationsMet',
  'trace_id': 'tr-6df53f787a6532119e207d72e55fd0c2',
  'span_id': 'dc6102b959144839',
  'source': {'source_type': 'LLM_JUDGE', 'source_id': 'openai:/gpt-4o-mini'},
  'create_time': '2026-07-25T00:53:57.524Z',
  'last_update_time': '2026-07-25T00:53:57.524Z',
  'feedback': {'value': 'no'},
  'rationale': 'The response does not clearly indicate a specific route to escalation, as it does not mention that the issue will be escalated or that it is being directed to the escalation team. Instead, it simply states that someone will contact the user, which could imply various levels of support without confirming that it is being escalated.',
  'metadata': {'mlflow.assessment.judgeCost': '9.915e-05',
   'guideline': '\n    You are evaluating whether an agent met specific expectations.\n    \n    Given the expectation criteria: {{ expectations }}\n    \n    Extract the text under the === RESPONSE === 

In [23]:
if failed_traces:
    failed_df = pd.DataFrame(failed_traces)
    display(failed_df)
else:
    print("No failed traces to display.")


,trace_id,request,response,reasons,reasons_list
0,tr-64bfeb7e9224be6dffcaa339a9aed1d9,"{'user_message': 'I have a headache, what medi...",=== INTENT ===\nescalation\n\n=== TOOLS ===\n[...,"Faithfulness Failed: The CONTEXT is empty, pro...","[{'scorer_name': 'Faithfulness', 'error_reason..."
1,tr-1ea5370a2fb844d649e0d223ac46be2b,{'user_message': 'What should I do if my arm h...,=== INTENT ===\nescalation\n\n=== TOOLS ===\n[...,ExpectationsMet Failed: The response does not ...,"[{'scorer_name': 'ExpectationsMet', 'error_rea..."
2,tr-ca34320f35072d86109da83f05a616f4,{'user_message': 'Can I book an appointment fo...,=== INTENT ===\nbooking\n\n=== TOOLS ===\n[]\n...,ExpectationsMet Failed: The response does not ...,"[{'scorer_name': 'ExpectationsMet', 'error_rea..."
3,tr-b6d1e906f83fa28b87870d536e70375d,{'user_message': 'How much does a dermatology ...,"=== INTENT ===\nfaq\n\n=== TOOLS ===\n[""search...",ExpectationsMet Failed: The response does not ...,"[{'scorer_name': 'ExpectationsMet', 'error_rea..."
4,tr-7d7a660d69ca8dc47bc4d76865e3a3fa,{'user_message': 'The 10am slot with Dr. Silva...,=== INTENT ===\nbooking\n\n=== TOOLS ===\n[]\n...,ExpectationsMet Failed: The agent did not ask ...,"[{'scorer_name': 'ExpectationsMet', 'error_rea..."
5,tr-943f3e2dc62ff8c7e751a862e78bb219,{'user_message': 'Can I reschedule my appointm...,=== INTENT ===\nbooking\n\n=== TOOLS ===\n[]\n...,ExpectationsMet Failed: The response does not ...,"[{'scorer_name': 'ExpectationsMet', 'error_rea..."
6,tr-b5fa726883ce28023b584ce73bcb6b29,{'user_message': '[System Event: Payment Expir...,=== INTENT ===\nescalation\n\n=== TOOLS ===\n[...,ExpectationsMet Failed: The response does not ...,"[{'scorer_name': 'ExpectationsMet', 'error_rea..."
7,tr-76b96b6342ab713aa946de49154ec572,"{'user_message': 'Yes, that's correct.', 'conv...",=== INTENT ===\nbooking\n\n=== TOOLS ===\n[]\n...,ExpectationsMet Failed: The response does not ...,"[{'scorer_name': 'ExpectationsMet', 'error_rea..."
8,tr-1496b12efceaa9bed4e685388f617a4b,"{'user_message': 'I need an endocrinologist.',...",=== INTENT ===\nbooking\n\n=== TOOLS ===\n[]\n...,"Faithfulness Failed: The CONTEXT is empty, mea...","[{'scorer_name': 'Faithfulness', 'error_reason..."
9,tr-c0a6294e652f23edb742847157d9a432,{'user_message': 'Do you have anything next Tu...,=== INTENT ===\nbooking\n\n=== TOOLS ===\n[]\n...,ExpectationsMet Failed: The response did not c...,"[{'scorer_name': 'ExpectationsMet', 'error_rea..."


In [24]:
failed_df['n_reasons'] = failed_df['reasons'].apply(lambda x: len(x.split("|")) )


In [25]:
failed_df['reasons'].values

array(['Faithfulness Failed: The CONTEXT is empty, providing no specific information or details to support the RESPONSE. The RESPONSE contains a message suggesting that someone will contact the individual for assistance, which is not grounded in any context present.',
       'ExpectationsMet Failed: The response does not address medical advice or recommend consulting a doctor. It simply states that someone will contact the user for assistance, which does not fulfill the expectation criteria. | Faithfulness Failed: The CONTEXT is empty, which means there are no supporting facts, claims, or information provided. The RESPONSE, however, gives a specific message indicating that someone will contact the user for assistance. Since the RESPONSE contains information that is not grounded in any provided CONTEXT, it cannot be considered completely supported.',
       'ExpectationsMet Failed: The response does not directly handle the booking. Instead, it asks for additional information to retrieve

In [26]:
failing_reasons_df = failed_df.explode('reasons_list')

In [27]:
all_failing_reasons = pd.concat((failing_reasons_df, failing_reasons_df.reasons_list.apply(pd.Series)), axis = 1)

In [28]:

all_failing_reasons['scorer_name'].value_counts()

scorer_name
Faithfulness       39
ExpectationsMet    33
ResponseQuality     5
Name: count, dtype: int64

In [29]:

all_failing_reasons['error_reason'].value_counts()

error_reason
no error rationale    77
Name: count, dtype: int64

In [41]:
all_failing_reasons['rationale'].value_counts().sort_values( ascending=False).to_frame()

,count
rationale,
"The CONTEXT is empty, providing no specific information or details to support the RESPONSE. The RESPONSE contains a message suggesting that someone will contact the individual for assistance, which is not grounded in any context present.",1
"The response does not address medical advice or recommend consulting a doctor. It simply states that someone will contact the user for assistance, which does not fulfill the expectation criteria.",1
"The CONTEXT is empty, which means there are no supporting facts, claims, or information provided. The RESPONSE, however, gives a specific message indicating that someone will contact the user for assistance. Since the RESPONSE contains information that is not grounded in any provided CONTEXT, it cannot be considered completely supported.",1
"The response does not directly handle the booking. Instead, it asks for additional information to retrieve the user's record, indicating that the agent is not processing the booking at that moment.",1
"The CONTEXT is empty (indicated by """"), which means there is no information to support any claims or statements made in the RESPONSE. The RESPONSE asks for personal information and implies an action regarding a booking without any grounding in the provided CONTEXT.",1
...,...
"The CONTEXT section is empty, which means there is no information provided to support or ground the claims made in the RESPONSE. As a result, the RESPONSE cannot be considered entirely supported by the CONTEXT.",1
"The response provided does not route to booking intent as it fails to ask for booking-related information or confirm a booking process. Instead, it only asks about registration, which is unrelated to the expectation of routing towards booking.",1
"The CONTEXT provided is empty (""""), which means there is no supporting information given. The RESPONSE, however, contains a question asking for personal information related to a registration system. Since the RESPONSE makes claims and requests that are not found or supported in any CONTEXT, it cannot be considered faithful to the provided context.",1


In [46]:
all_failing_reasons[['trace_id','scorer_name','rationale']]

,trace_id,scorer_name,rationale
0,tr-64bfeb7e9224be6dffcaa339a9aed1d9,Faithfulness,"The CONTEXT is empty, providing no specific in..."
1,tr-1ea5370a2fb844d649e0d223ac46be2b,ExpectationsMet,The response does not address medical advice o...
1,tr-1ea5370a2fb844d649e0d223ac46be2b,Faithfulness,"The CONTEXT is empty, which means there are no..."
2,tr-ca34320f35072d86109da83f05a616f4,ExpectationsMet,The response does not directly handle the book...
2,tr-ca34320f35072d86109da83f05a616f4,Faithfulness,"The CONTEXT is empty (indicated by """"), which ..."
...,...,...,...
40,tr-aa12a8cac390dd9625b24798c5265e13,Faithfulness,"The CONTEXT section is empty, which means ther..."
41,tr-1902961650d31b8d58d2a1fbdd342f50,ExpectationsMet,The response provided does not route to bookin...
41,tr-1902961650d31b8d58d2a1fbdd342f50,Faithfulness,"The CONTEXT provided is empty (""""), which mean..."
42,tr-6df53f787a6532119e207d72e55fd0c2,ExpectationsMet,The response does not clearly indicate a speci...


In [47]:
all_failing_reasons['expectation'].value_counts()


Series([], Name: count, dtype: int64)

----
---

In [44]:
failed_traces

[{'trace_id': 'tr-64bfeb7e9224be6dffcaa339a9aed1d9',
  'request': {'user_message': 'I have a headache, what medicine should I take?',
   'conversation_history': [],
   'agent_under_test': 'faq',
   'kwargs': {}},
  'response': '=== INTENT ===\nescalation\n\n=== TOOLS ===\n[]\n\n=== CONTEXT ===\n""\n\n=== RESPONSE ===\nThank you for reaching out. This requires personal attention from one of our staff members. Someone will contact you shortly to assist you directly. We appreciate your patience.\n',
  'reasons': 'Faithfulness Failed: The CONTEXT is empty, providing no specific information or details to support the RESPONSE. The RESPONSE contains a message suggesting that someone will contact the individual for assistance, which is not grounded in any context present.',
  'reasons_list': [{'scorer_name': 'Faithfulness',
    'error_reason': 'no error rationale',
    'rationale': 'The CONTEXT is empty, providing no specific information or details to support the RESPONSE. The RESPONSE contain

In [15]:
from IPython.display import display

if failed_traces:
    # Display the full interactive trace for the first failed trace
    first_failed_trace_id = failed_traces[0]['trace_id']
    print(f"\nDisplaying full interactive trace for Trace ID: {first_failed_trace_id}")
    
    try:
        trace_obj = client.get_trace(first_failed_trace_id)
        display(trace_obj)
    except Exception as e:
        print(f"Could not fetch trace {first_failed_trace_id} from DB: {e}")



Displaying full interactive trace for Trace ID: tr-cda9fc6533486aa22c7962fbc1af85fb


Trace(trace_id=tr-cda9fc6533486aa22c7962fbc1af85fb)

In [ ]:
import mlflow

mlflow.set_tracking_uri("sqlite:///evaluation/mlflow.db")

# Replace with the actual Run ID printed in your terminal
run_id = "YOUR_RUN_ID" 

# This loads the exact same table you see in the CSV directly from MLflow!
df = mlflow.load_table(
    "eval_results", 
    run_ids=[run_id]
)

print(df.head())
